# 01. Xor tricks

**Tier:** Deep. **Why you'd use this:** Pairs of equal values cancel under XOR. One linear pass can recover the value that appears once when every other value appears twice, with constant extra memory.

**After this lesson you can:** use XOR to find a singleton among pairs; read bits with shifts and masks when counts matter.
About 35 minutes. **Uses:** [Loops](../../course/03-control-flow/02-loops-lesson.ipynb), [Lists](../../course/04-collections/01-lists-lesson.ipynb)


## Quick review

1. What happens when you XOR a value with itself?
2. Why can XOR find a value that appears once among pairs?

<details><summary>Answers</summary>

`x ^ x` is `0`. XOR is associative and commutative, and `x ^ 0` is `x`, so every paired value cancels and the singleton remains.
</details>


## Recognize it

Every value appears twice except one that appears once, and you must find that singleton in linear time with constant extra memory. The prompt says "single number", "appears once", or "pairs cancel".

Signals: even counts cancel, one odd count remains, and the answer is a bit-aggregation rather than a hash map of frequencies. A look-alike that is not this pattern: counting how many bits are set across a range of integers — that is a DP or shift recurrence, not a pairwise XOR cancel. Finding two singletons needs an extra bit-partition step before you can finish with XOR on each half.


## Why it works this way

XOR is bitwise addition without carry. Bit by bit, two ones cancel to zero. For whole integers that means `x ^ x = 0` and `x ^ 0 = x`. XOR is associative and commutative, so the order of values does not matter: pairing each duplicate cancels that value out of the running accumulator.

Start from `0`. Fold `acc = acc ^ value` across the list. Every value that appears twice contributes two identical XORs and disappears. The value that appears once remains as `acc`. Time is one pass over `n` values. Extra memory is a single integer. A hash map of counts also finds the singleton but uses memory proportional to the number of distinct values; XOR wins when the problem forbids that space.

The same algebra underpins nearby tricks. Missing number in `0..n` XORs every index with every value and with `n`. Parity of a bit across a list is the XOR of that bit. Shifts (`>>`, `<<`) and masks (`& 1`) read individual bits when you need counts rather than cancels. Practice uses that shift idea: `popcount[i]` reuses `popcount[i >> 1]` and adds the lowest bit.

Draw a tiny trace when the idea feels abstract. For `[2, 2, 1]` the accumulator moves `0 → 2 → 0 → 1`. The middle zero is not a bug; it is the pair canceling before the singleton arrives. Negatives work the same way in two's complement: XOR still cancels equal bit patterns.

XOR does not help when frequencies are three or five unless you track bits with a different modulus. Sorting and scanning adjacent equals works but costs `n log n` time. Prefer a set of "seen once" when you need the values themselves for a follow-up step, not only the singleton identity.


## When to use it

Use a running XOR when every element appears twice except one singleton, or when you need the parity of bits across a stream. Use a hash map when frequencies are arbitrary or you must report every duplicate. Use the shift recurrence `popcount[i] = popcount[i >> 1] + (i & 1)` when you need set-bit counts for every integer through `n`. Skip XOR when the prompt needs the positions of duplicates, not only the unpaired value, or when more than one value appears an odd number of times without an extra partition step.


## Brute force, and why it's slow

For each value, scan the whole list to count occurrences. Count the membership tests.


In [1]:
def single_number_slow(nums):
    tests = 0
    for value in nums:
        count = 0
        for other in nums:
            tests += 1
            if other == value:
                count += 1
        if count == 1:
            return value, tests
    return None, tests


for size in (21, 41, 81):
    data = [i // 2 for i in range(size - 1)] + [size]
    answer, tests = single_number_slow(data)
    print(size, answer, tests)

21 21 441
41 41 1681
81 81 6561


## Key insight

`x ^ x = 0` and `x ^ 0 = x`. Fold XOR across the list; paired values cancel and the singleton remains in the accumulator.


## Worked example

Every value appears twice except one. Return the value that appears once.


In [2]:
def single_number_bit(nums: list[int]) -> int:
    # 1. Start from identity for XOR.
    acc = 0
    # 2. Fold: pairs cancel, the singleton remains.
    for value in nums:
        acc = acc ^ value
    return acc


print(single_number_bit([2, 2, 1]))
print(single_number_bit([4, 1, 2, 1, 2]))

1
4


## Trace

`[4, 1, 2, 1, 2]`:

| value | acc after step |
|---|---|
| 4 | `0 ^ 4 = 4` |
| 1 | `4 ^ 1 = 5` |
| 2 | `5 ^ 2 = 7` |
| 1 | `7 ^ 1 = 6` |
| 2 | `6 ^ 2 = 4` |

Answer **4**.


## Cost

One pass: time grows with `n`. Extra memory is one integer. That beats the nested count because each value is mixed into the accumulator once instead of being compared to every other value. The constant factors stay tiny: XOR is a single CPU-level op per element.


## Variants

Practice fills popcounts through `n` with a shift recurrence. Related challenges (not solved here): [Single Number](../../challenges/problems/66-single-number.ipynb), [Counting Bits](../../challenges/problems/67-counting-bits.ipynb).


## Common mistake

This bug is silent. The code uses `+` instead of `^`, so it sums the list instead of canceling pairs. For `[2, 2, 1]` the print is `5`, and the answer is `1`.


In [3]:
def single_number_wrong(nums):
    acc = 0
    for value in nums:
        acc = acc + value  # sum, not XOR
    return acc


print(single_number_wrong([2, 2, 1]))

5


The fix uses `^`. Pairs cancel and the singleton remains. The worked example prints `1` for that input.


In [4]:
print(single_number_bit([2, 2, 1]))

1


## Python idioms

`functools.reduce` with `operator.xor` is the same fold in one expression. `^`, `&`, `|`, `<<`, and `>>` are the bit operators; prefer them over stringifying with `bin` when you need arithmetic on bits.


In [5]:
import functools
import operator


def single_number_reduce(nums: list[int]) -> int:
    return functools.reduce(operator.xor, nums, 0)


print(single_number_reduce([4, 1, 2, 1, 2]))

4


## Recap

- XOR cancels equal pairs and leaves the singleton.
- One accumulator and one pass are enough.
- Use shifts and masks when you need bit counts, not cancels.

Next: [practice](01-xor-tricks-practice.ipynb).
